In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Load the graph
import networkx as nx
G = nx.read_gml('../data/graph/LEMD_EGLL_2023_04_01.gml')
# Load node ID mappings
node_list = list(G.nodes())
node_to_idx = {node: idx for idx, node in enumerate(node_list)}
idx_to_node = {idx: node for node, idx in node_to_idx.items()}

In [4]:
# Shortest path from LEMD to EGLL
shortest_path = nx.shortest_path(G, source='LEMD', target='EGLL')
print(shortest_path)

['LEMD', 'LETP', 'KOVAK', 'EGLL']


In [5]:
# Load the value functions
import numpy as np
V_final = np.load('../data/results/forward/V_final.npy')
eta_final = np.load('../data/results/forward/eta_final.npy')
alt_final = np.load('../data/results/forward/alt_final.npy')
phase_final = np.load('../data/results/forward/phase_final.npy')
V_final_margin = np.array([np.sum(row[np.isfinite(row)]) for row in V_final])


# Inspection

In [6]:
V_final_margin[node_to_idx['EGLL']]

np.float64(-1490.6529012739747)

# Cost Calculation

In [35]:
from equinox.sampling.sample_from_v import get_route_cost 
from equinox.wind.wind_date import WindDate

takeoff = "2023-04-01 12:00:00"
graph_file = '../data/graph/LEMD_EGLL_2023_04_01.gml'
dist_matrix_file = '../data/graph/LEMD_EGLL_2023_04_01_distances.npy'
ac_matrix_file = '../data/graph/LEMD_EGLL_2023_04_01_charges.npy'
wind_model = WindDate(date_str="2024-04-01", data_dir='../data/era5')

wind_dir = '../data/era5'

total_cost = get_route_cost(
    route=shortest_path,
    takeoff_time_str=takeoff,
    graph_path=graph_file,
    dist_matrix_path=dist_matrix_file,
    ac_matrix_path=ac_matrix_file,
    wind_model=wind_model,
    source_elevation_ft=0,
    destination_elevation_ft=0
)

Using device: cpu

--- Forward State Prediction ---
Takeoff from LEMD at 2023-04-01 12:00:00 alt 0 ft, phase 0
  Segment LEMD -> LETP:
    Tailwind at LEMD (used for cost): 7.33 kts
    Arrival at LETP: ETA 2023-04-01 12:08:27, Alt 20905 ft, Phase 0
  Segment LETP -> KOVAK:
    Tailwind at LETP (used for cost): 4.95 kts
    Arrival at KOVAK: ETA 2023-04-01 13:06:59, Alt 35000 ft, Phase 1
  Segment KOVAK -> EGLL:
    Tailwind at KOVAK (used for cost): 13.57 kts
    Arrival at EGLL: ETA 2023-04-01 13:34:15, Alt 35000 ft, Phase 1

--- Calculating Edge Costs ---
  Cost for edge LEMD -> LETP (tailwind 7.33 kts): 0.05
  Cost for edge LETP -> KOVAK (tailwind 4.95 kts): 0.63
  Cost for edge KOVAK -> EGLL (tailwind 13.57 kts): 0.33

Total calculated route cost: 1.01


# Arrival at a Node

In [47]:
sumexp = 0
for succ in LEMD_successors:
    V_succ = -V_final_margin[node_to_idx[succ]]
    sumexp += np.exp(V_succ)
print(-np.log(sumexp))

-214.83217448674188


## Verify the first Propagation

In [16]:
LEMD_successors = list(G.successors('LEMD'))
print(f'There are {len(LEMD_successors)} successors to LEMD')

There are 67 successors to LEMD


In [26]:
import torch
from equinox.sampling.sample_from_v import get_wind, get_next_state_fw
PHASE_CLIMB = 0
PHASE_CRUISE = 1
PHASE_DESCENT = 2
MPS_TO_KNOTS = 1.9438444924406047

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def state_propagation_wrapper(u_node_id, v_node_id, current_alt_ft, current_eta_ssm, current_phase,
                              climb_perf_table, descent_perf_table, wind_model, node_to_idx_for_matrix):
    coords_src_list = (G.nodes[u_node_id]['lat'], G.nodes[u_node_id]['lon'])
    coords_tgt_list = (G.nodes[v_node_id]['lat'], G.nodes[v_node_id]['lon'])

    coords_src = torch.tensor([coords_src_list], dtype=torch.float32, device=device)
    coords_tgt = torch.tensor([coords_tgt_list], dtype=torch.float32, device=device)

    active_performance_table = climb_perf_table
    if current_phase.item() == PHASE_DESCENT:
        active_performance_table = descent_perf_table

    next_alt_ft, next_eta_ssm, next_phase = get_next_state_fw(
        coords_src=coords_src, alts_src=current_alt_ft, eta_src=current_eta_ssm,
        phase_src=current_phase, coords_tgt=coords_tgt,
        climb_performance=active_performance_table, wind_model=wind_model
    )

    return next_alt_ft, next_eta_ssm, next_phase

In [27]:
from equinox.vnav.vnav_performance import Performance, get_eta_and_distance_climb
from equinox.vnav.vnav_profiles_rev1 import NARROW_BODY_JET_CLIMB_PROFILE, NARROW_BODY_JET_DESCENT_PROFILE, NARROW_BODY_JET_CLIMB_VS_PROFILE, NARROW_BODY_JET_DESCENT_VS_PROFILE
narrow_body_jet_performance = Performance(
    climb_speed_profile=NARROW_BODY_JET_CLIMB_PROFILE,
    descent_speed_profile=NARROW_BODY_JET_DESCENT_PROFILE,
    climb_vertical_speed_profile=NARROW_BODY_JET_CLIMB_VS_PROFILE,
    descent_vertical_speed_profile=NARROW_BODY_JET_DESCENT_VS_PROFILE,
    cruise_altitude_ft=35000.0,
    cruise_speed_kts=450.0
)

climb_perf_table = get_eta_and_distance_climb(
    narrow_body_jet_performance,
    origin_airport_elevation_ft=0
)

In [28]:
wind_model = WindDate(date_str="2024-04-01", data_dir='../data/era5')

In [29]:
# Load the distance matrix
dist_matrix = np.load('../data/graph/LEMD_EGLL_2023_04_01_distances.npy')

# Load the airspace charges matrix
ac_matrix = np.load('../data/graph/LEMD_EGLL_2023_04_01_charges.npy')

node_to_idx = {node_id: i for i, node_id in enumerate(G.nodes())}

In [40]:
for succ in LEMD_successors:
    u = 'LEMD'
    v = succ 
    current_alt_ft = torch.tensor([0], dtype=torch.float32, device=device)
    current_eta_ssm = torch.tensor([12 * 3600], dtype=torch.float32, device=device)
    current_phase = torch.tensor([PHASE_CLIMB], dtype=torch.int32, device=device)
    route_edges_params_for_cost = []
    next_alt_ft, next_eta_ssm, next_phase = state_propagation_wrapper(
        u, v, current_alt_ft, current_eta_ssm, current_phase, climb_perf_table, None, wind_model, node_to_idx
    )
    next_alt_ft = next_alt_ft.detach().cpu().numpy().item()
    next_eta_ssm = next_eta_ssm.detach().cpu().numpy().item()
    next_phase = next_phase.detach().cpu().numpy().item()
    print(f"{succ}: alt_ft={next_alt_ft}, eta_ssm={next_eta_ssm}, phase={next_phase}")

RBO: alt_ft=14378.0244140625, eta_ssm=43531.33984375, phase=0
RBO_16: alt_ft=11637.7451171875, eta_ssm=43449.1328125, phase=0
LECV: alt_ft=10883.4677734375, eta_ssm=43426.50390625, phase=0
LERM: alt_ft=14575.470703125, eta_ssm=43537.265625, phase=0
LETP: alt_ft=20473.103515625, eta_ssm=43714.19140625, phase=0
PINAR: alt_ft=23040.55859375, eta_ssm=43791.21875, phase=0
EDIGO_39: alt_ft=22708.509765625, eta_ssm=43781.25390625, phase=0
OSTIX: alt_ft=26759.8359375, eta_ssm=43902.796875, phase=0
GASMO: alt_ft=29448.4453125, eta_ssm=44012.421875, phase=0
SIE_06: alt_ft=18397.595703125, eta_ssm=43651.9296875, phase=0
SIE: alt_ft=19451.00390625, eta_ssm=43683.53125, phase=0
UNSOL_45: alt_ft=23396.650390625, eta_ssm=43801.8984375, phase=0
DISKO_38: alt_ft=19963.29296875, eta_ssm=43698.8984375, phase=0
SEGRE: alt_ft=25984.439453125, eta_ssm=43879.53125, phase=0
LETO: alt_ft=3750.436279296875, eta_ssm=43275.0078125, phase=0
BASIM_84: alt_ft=23040.2890625, eta_ssm=43791.20703125, phase=0
ZANKO: alt